In [2]:
import pandas as pd
import numpy as np
import uuid

In [18]:
# Number of records
num_rows = 5000

np.random.seed(42)

data = []

In [19]:
for _ in range(num_rows):
    # 1. Base Student Features
    attendance = np.random.uniform(65, 100) # Bumped minimum attendance up slightly
    latent_ability = np.random.normal(0.70, 0.12) # Slightly tighter, higher mean ability
    latent_ability = np.clip(latent_ability, 0.1, 1.0)

    # 2. Performance Columns (Tied to ability and attendance)
    quiz_avg = np.clip(
        latent_ability * 75 + attendance * 0.15 + np.random.normal(0, 5),
        0, 100
    )

    assignment_avg = np.clip(
        latent_ability * 78 + attendance * 0.12 + np.random.normal(0, 5),
        0, 100
    )

    midterm_score = np.clip(
        latent_ability * 80 + attendance * 0.10 + np.random.normal(0, 6),
        0, 100
    )

    historical_gpa = round(
        np.clip(latent_ability * 3.5 + np.random.normal(0.5, 0.3), 2.0, 4.0),
        2
    )

    # Reality check: most students study 5-25 hours, not a completely flat 1-40
    study_hours = round(
        np.clip(latent_ability * 20 + np.random.uniform(2, 15), 1, 40),
        1
    )

    participation_score = np.clip(
        np.random.normal(80, 8), # Higher average participation
        40, 100
    )

    subject_difficulty = round(
        np.random.uniform(1, 10),
        2
    )

    semester_number = np.random.randint(1, 9)

    # 3. Target (Final Score) - Recalibrated weights to hit an ~80% pass rate
    final_score = (
        attendance * 0.15 +          # Max: 15
        quiz_avg * 0.20 +            # Max: 20
        assignment_avg * 0.20 +      # Max: 20
        midterm_score * 0.25 +       # Max: 25
        (historical_gpa * 2.5) * 0.05 + # Max: 0.5 (scaled to reduce GPA dominance)
        (study_hours / 40) * 10 +    # Max: 10
        participation_score * 0.10   # Max: 10
    )

    # Subtract difficulty penalty gently (Max penalty now ~4 points instead of 15)
    final_score -= (subject_difficulty * 0.4)

    # Add a touch of natural variance
    final_score += np.random.normal(0, 3)
    final_score = np.clip(final_score, 0, 100)

    # 4. Target Classification (60 is Passing)
    pass_fail = 1 if final_score >= 60 else 0

    data.append([
        str(uuid.uuid4()),
        round(attendance, 2),
        round(quiz_avg, 2),
        round(assignment_avg, 2),
        round(midterm_score, 2),
        historical_gpa,
        study_hours,
        round(participation_score, 2),
        subject_difficulty,
        semester_number,
        round(final_score, 2),
        pass_fail
    ])

In [20]:
columns = [
    "student_id",
    "attendance_percentage",
    "quiz_score_avg",
    "assignment_score_avg",
    "midterm_score",
    "historical_gpa",
    "study_hours_per_week",
    "participation_score",
    "subject_difficulty_score",
    "semester_number",
    "final_score",
    "pass_fail"
]

In [21]:
df = pd.DataFrame(data, columns=columns)

In [22]:
df.head()

,student_id,attendance_percentage,quiz_score_avg,assignment_score_avg,midterm_score,historical_gpa,study_hours_per_week,participation_score,subject_difficulty_score,semester_number,final_score,pass_fail
0,510845f3-fec3-423e-84ba-12422f4f8058,78.11,55.80,54.96,59.20,2.31,15.7,75.80,3.74,6,51.65,0
1,85fe2085-7fc6-41a7-9819-9062e3f12102,66.63,69.04,64.40,69.31,3.19,23.5,83.34,5.05,2,69.10,1
2,a1e6e199-b45e-408a-b633-2cc2656dab41,98.21,60.37,57.75,71.17,2.69,15.0,71.54,9.18,4,65.41,1
3,ed75ac0f-5371-4900-b7fa-b59e8978a2ec,79.88,59.76,50.62,44.16,2.34,26.4,72.74,6.13,4,60.38,1
4,2bcfaf42-2ff8-45b6-9975-e8f668e78fe8,66.58,45.49,57.01,49.74,2.12,20.6,72.96,2.27,7,49.10,0


In [23]:
df.shape

(5000, 12)

In [24]:
df.describe()


,attendance_percentage,quiz_score_avg,assignment_score_avg,midterm_score,historical_gpa,study_hours_per_week,participation_score,subject_difficulty_score,semester_number,final_score,pass_fail
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,82.586242,64.731406,64.448524,64.144888,2.951908,22.481380,79.999934,5.526170,4.522000,66.019992,0.766600
std,10.104653,10.460389,10.759124,11.381573,0.491422,4.486698,7.900784,2.602392,2.292022,8.226966,0.423037
min,65.000000,29.580000,24.190000,24.110000,2.000000,8.100000,49.940000,1.000000,1.000000,38.030000,0.000000
25%,73.847500,57.680000,57.210000,56.507500,2.600000,19.100000,74.730000,3.280000,3.000000,60.397500,1.000000
50%,82.435000,64.460000,64.420000,64.350000,2.950000,22.500000,80.045000,5.570000,4.000000,65.860000,1.000000
75%,91.415000,71.765000,71.562500,71.705000,3.290000,25.900000,85.350000,7.780000,7.000000,71.430000,1.000000
max,99.990000,99.780000,98.150000,100.000000,4.000000,34.800000,100.000000,10.000000,8.000000,93.860000,1.000000


In [35]:
import pandas as pd
import numpy as np

def check_outliers_iqr(df):
    """
    Scans a dataframe and prints the number of outliers found 
    in each numeric column using the IQR technique.
    """
    print("--- Outlier Detection Analysis (IQR Method) ---")
    
    # Select only numeric columns (ignores UUIDs or categorical strings)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        # Step 1: Calculate Q1 (25th percentile) and Q3 (75th percentile)
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        
        # Step 2: Calculate the Interquartile Range (IQR)
        iqr = q3 - q1
        
        # Step 3: Define the cut-off boundaries
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        # Step 4: Identify rows outside the boundaries
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        
        # Step 5: Print the results
        print(f"{col}: {len(outliers)} outliers (Lower: {lower_bound:.2f}, Upper: {upper_bound:.2f})")

# --- HOW TO USE IT WITH YOUR DATA ---
# Assuming 'data' is the list from your loop:
columns = [
    "student_id", "attendance_percentage", "quiz_score_avg", "assignment_score_avg", 
    "midterm_score", "historical_gpa", "study_hours_per_week", "participation_score", 
    "subject_difficulty_score", "semester_number", "final_score", "pass_fail"
]

# Convert your data list into a Pandas DataFrame
df = pd.DataFrame(data, columns=columns)

# Run the function
check_outliers_iqr(df)

--- Outlier Detection Analysis (IQR Method) ---
attendance_percentage: 0 outliers (Lower: 47.50, Upper: 117.77)
quiz_score_avg: 32 outliers (Lower: 36.55, Upper: 92.89)
assignment_score_avg: 38 outliers (Lower: 35.68, Upper: 93.09)
midterm_score: 34 outliers (Lower: 33.71, Upper: 94.50)
historical_gpa: 0 outliers (Lower: 1.57, Upper: 4.33)
study_hours_per_week: 2 outliers (Lower: 8.90, Upper: 36.10)
participation_score: 18 outliers (Lower: 58.80, Upper: 101.28)
subject_difficulty_score: 0 outliers (Lower: -3.47, Upper: 14.53)
semester_number: 0 outliers (Lower: -3.00, Upper: 13.00)
final_score: 40 outliers (Lower: 43.85, Upper: 87.98)
pass_fail: 1167 outliers (Lower: 1.00, Upper: 1.00)


In [27]:
##checking the outliers
import pandas as pd
import matplotlib.pyplot as plt



numerical_cols = [
    'attendance_percentage',
    'quiz_score_avg',
    'assignment_score_avg',
    'midterm_score',
    'historical_gpa',
    'study_hours_per_week',
    'participation_score',
    'subject_difficulty_score',
    'semester_number',
    'final_score'
]

for col in numerical_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ]

    print(f"{col}: {len(outliers)} outliers")

attendance_percentage: 0 outliers
quiz_score_avg: 32 outliers
assignment_score_avg: 38 outliers
midterm_score: 34 outliers
historical_gpa: 0 outliers
study_hours_per_week: 2 outliers
participation_score: 18 outliers
subject_difficulty_score: 0 outliers
semester_number: 0 outliers
final_score: 40 outliers


In [28]:
# adding the clip to for handling the outlier
df['study_hours_per_week'] = df['study_hours_per_week'].clip(lower=8.90, upper=36.10)

# 2. Cap Participation Score (Lower: 58.80, Upper: 101.28)
df['participation_score'] = df['participation_score'].clip(lower=58.80, upper=101.28)

# 3. Cap Academic Performance Scores based on your IQR fences
df['quiz_score_avg'] = df['quiz_score_avg'].clip(lower=36.55, upper=92.89)
df['assignment_score_avg'] = df['assignment_score_avg'].clip(lower=35.68, upper=93.09)
df['midterm_score'] = df['midterm_score'].clip(lower=33.71, upper=94.50)
df['final_score'] = df['final_score'].clip(lower=43.85, upper=87.98)

# 4. Recalculate pass_fail based on the newly capped final_score to maintain integrity
df['pass_fail'] = np.where(df['final_score'] >= 60, 1, 0)

In [29]:
df.head()

,student_id,attendance_percentage,quiz_score_avg,assignment_score_avg,midterm_score,historical_gpa,study_hours_per_week,participation_score,subject_difficulty_score,semester_number,final_score,pass_fail
0,510845f3-fec3-423e-84ba-12422f4f8058,78.11,55.80,54.96,59.20,2.31,15.7,75.80,3.74,6,51.65,0
1,85fe2085-7fc6-41a7-9819-9062e3f12102,66.63,69.04,64.40,69.31,3.19,23.5,83.34,5.05,2,69.10,1
2,a1e6e199-b45e-408a-b633-2cc2656dab41,98.21,60.37,57.75,71.17,2.69,15.0,71.54,9.18,4,65.41,1
3,ed75ac0f-5371-4900-b7fa-b59e8978a2ec,79.88,59.76,50.62,44.16,2.34,26.4,72.74,6.13,4,60.38,1
4,2bcfaf42-2ff8-45b6-9975-e8f668e78fe8,66.58,45.49,57.01,49.74,2.12,20.6,72.96,2.27,7,49.10,0


In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   student_id                5000 non-null   str    
 1   attendance_percentage     5000 non-null   float64
 2   quiz_score_avg            5000 non-null   float64
 3   assignment_score_avg      5000 non-null   float64
 4   midterm_score             5000 non-null   float64
 5   historical_gpa            5000 non-null   float64
 6   study_hours_per_week      5000 non-null   float64
 7   participation_score       5000 non-null   float64
 8   subject_difficulty_score  5000 non-null   float64
 9   semester_number           5000 non-null   int64  
 10  final_score               5000 non-null   float64
 11  pass_fail                 5000 non-null   int64  
dtypes: float64(9), int64(2), str(1)
memory usage: 468.9 KB


In [31]:
df.describe()


,attendance_percentage,quiz_score_avg,assignment_score_avg,midterm_score,historical_gpa,study_hours_per_week,participation_score,subject_difficulty_score,semester_number,final_score,pass_fail
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,82.586242,64.731170,64.459896,64.154960,2.951908,22.481620,80.010418,5.526170,4.522000,66.019990,0.766800
std,10.104653,10.420514,10.694335,11.316793,0.491422,4.485954,7.869068,2.602392,2.292022,8.178384,0.422911
min,65.000000,36.550000,35.680000,33.710000,2.000000,8.900000,58.800000,1.000000,1.000000,43.850000,0.000000
25%,73.847500,57.680000,57.210000,56.507500,2.600000,19.100000,74.730000,3.280000,3.000000,60.397500,1.000000
50%,82.435000,64.460000,64.420000,64.350000,2.950000,22.500000,80.045000,5.570000,4.000000,65.860000,1.000000
75%,91.415000,71.765000,71.562500,71.705000,3.290000,25.900000,85.350000,7.780000,7.000000,71.430000,1.000000
max,99.990000,92.890000,93.090000,94.500000,4.000000,34.800000,100.000000,10.000000,8.000000,87.980000,1.000000


In [34]:
import pandas as pd
import numpy as np

def check_outliers_iqr_filtered(df):
    """
    Scans a dataframe and prints the number of outliers found,
    explicitly restricting the check to continuous numerical score columns.
    """
    print("--- Outlier Detection Analysis (Continuous Columns Only) ---")
    
    # Explicitly list only the continuous columns to check
    continuous_cols = [
        "attendance_percentage", 
        "quiz_score_avg", 
        "assignment_score_avg", 
        "midterm_score", 
        "historical_gpa", 
        "study_hours_per_week", 
        "participation_score", 
        "final_score"
    ]
    
    # Ensure columns exist in the DataFrame before looping
    valid_cols = [col for col in continuous_cols if col in df.columns]
    
    for col in valid_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        
        print(f"{col}: {len(outliers)} outliers (Lower: {lower_bound:.2f}, Upper: {upper_bound:.2f})")

# --- HOW TO RUN IT ---
check_outliers_iqr_filtered(df)

--- Outlier Detection Analysis (Continuous Columns Only) ---
attendance_percentage: 0 outliers (Lower: 47.50, Upper: 117.77)
quiz_score_avg: 0 outliers (Lower: 36.55, Upper: 92.89)
assignment_score_avg: 0 outliers (Lower: 35.68, Upper: 93.09)
midterm_score: 0 outliers (Lower: 33.71, Upper: 94.50)
historical_gpa: 0 outliers (Lower: 1.57, Upper: 4.33)
study_hours_per_week: 0 outliers (Lower: 8.90, Upper: 36.10)
participation_score: 0 outliers (Lower: 58.80, Upper: 101.28)
final_score: 0 outliers (Lower: 43.85, Upper: 87.98)


In [37]:
import pandas as pd
import numpy as np

# List of continuous columns that require capping
continuous_cols = [
    "attendance_percentage", 
    "quiz_score_avg", 
    "assignment_score_avg", 
    "midterm_score", 
    "historical_gpa", 
    "study_hours_per_week", 
    "participation_score", 
    "final_score"
]

print("--- Applying IQR Capping to Continuous Columns ---")

for col in continuous_cols:
    if col in df.columns:
        # Calculate IQR and fences dynamically
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        
        lower_fence = q1 - 1.5 * iqr
        upper_fence = q3 + 1.5 * iqr
        
        # Apply the clip operation using the calculated fences
        df[col] = df[col].clip(lower=lower_fence, upper=upper_fence)
        
        print(f"Capped {col: <25} -> Min: {lower_fence:.2f}, Max: {upper_fence:.2f}")

# Recalculate pass_fail alignment based on the newly capped final_score
df['pass_fail'] = np.where(df['final_score'] >= 60, 1, 0)

print("\nCapping complete. Target pass_fail variable successfully updated.")

--- Applying IQR Capping to Continuous Columns ---
Capped attendance_percentage     -> Min: 47.50, Max: 117.77
Capped quiz_score_avg            -> Min: 36.55, Max: 92.89
Capped assignment_score_avg      -> Min: 35.68, Max: 93.09
Capped midterm_score             -> Min: 33.71, Max: 94.50
Capped historical_gpa            -> Min: 1.57, Max: 4.33
Capped study_hours_per_week      -> Min: 8.90, Max: 36.10
Capped participation_score       -> Min: 58.80, Max: 101.28
Capped final_score               -> Min: 43.85, Max: 87.98

Capping complete. Target pass_fail variable successfully updated.


In [38]:
import pandas as pd
import numpy as np

def check_outliers_iqr_filtered(df):
    """
    Scans a dataframe and prints the number of outliers found,
    explicitly restricting the check to continuous numerical score columns.
    """
    print("--- Outlier Detection Analysis (Continuous Columns Only) ---")
    
    # Explicitly list only the continuous columns to check
    continuous_cols = [
        "attendance_percentage", 
        "quiz_score_avg", 
        "assignment_score_avg", 
        "midterm_score", 
        "historical_gpa", 
        "study_hours_per_week", 
        "participation_score", 
        "final_score"
    ]
    
    # Ensure columns exist in the DataFrame before looping
    valid_cols = [col for col in continuous_cols if col in df.columns]
    
    for col in valid_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        
        print(f"{col}: {len(outliers)} outliers (Lower: {lower_bound:.2f}, Upper: {upper_bound:.2f})")

# --- HOW TO RUN IT ---
check_outliers_iqr_filtered(df)

--- Outlier Detection Analysis (Continuous Columns Only) ---
attendance_percentage: 0 outliers (Lower: 47.50, Upper: 117.77)
quiz_score_avg: 0 outliers (Lower: 36.55, Upper: 92.89)
assignment_score_avg: 0 outliers (Lower: 35.68, Upper: 93.09)
midterm_score: 0 outliers (Lower: 33.71, Upper: 94.50)
historical_gpa: 0 outliers (Lower: 1.57, Upper: 4.33)
study_hours_per_week: 0 outliers (Lower: 8.90, Upper: 36.10)
participation_score: 0 outliers (Lower: 58.80, Upper: 101.28)
final_score: 0 outliers (Lower: 43.85, Upper: 87.98)


In [39]:
df.describe()

,attendance_percentage,quiz_score_avg,assignment_score_avg,midterm_score,historical_gpa,study_hours_per_week,participation_score,subject_difficulty_score,semester_number,final_score,pass_fail
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,82.586242,64.731186,64.459905,64.154969,2.951908,22.481620,80.010418,5.526170,4.522000,66.019980,0.766800
std,10.104653,10.420520,10.694330,11.316786,0.491422,4.485954,7.869068,2.602392,2.292022,8.178383,0.422911
min,65.000000,36.552500,35.681250,33.711250,2.000000,8.900000,58.800000,1.000000,1.000000,43.848750,0.000000
25%,73.847500,57.680000,57.210000,56.507500,2.600000,19.100000,74.730000,3.280000,3.000000,60.397500,1.000000
50%,82.435000,64.460000,64.420000,64.350000,2.950000,22.500000,80.045000,5.570000,4.000000,65.860000,1.000000
75%,91.415000,71.765000,71.562500,71.705000,3.290000,25.900000,85.350000,7.780000,7.000000,71.430000,1.000000
max,99.990000,92.892500,93.091250,94.501250,4.000000,34.800000,100.000000,10.000000,8.000000,87.978750,1.000000


In [40]:
df.to_csv("../data/raw/student_performance_dataset.csv", index=False)

In [41]:
df.to_csv("../data/raw/student_performance_datasetv1.csv", index=False)